In [69]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,log_loss
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [70]:
df = pd.read_csv('../0.dataset/dataset_Kinerja_Karyawan_clean.csv')
df_x = df.drop(columns='Rating_Kinerja')
df_y = df['Rating_Kinerja']

# 1.Multi Classification Naive Bayes 

In [71]:
X_train,X_test,y_train,y_test = train_test_split(df_x,df_y,test_size=0.2,random_state=42)

In [72]:
params = {
   'alpha': [ 0.0001, 0.001, 0.01, 0.1],
    'fit_prior': [True, False]
}

cnb = CategoricalNB()
random_search = RandomizedSearchCV(estimator=cnb,param_distributions=params,n_iter=20,
                                   cv=6,scoring='accuracy',random_state=42,n_jobs=-1)
random_search.fit(X_train,y_train)
best_model_cnb = random_search.best_estimator_

print(f'\nHyperparameter Terbaik: {random_search.best_params_}')
print(f'\nAkurasi Validasi Terbaik: {random_search.best_score_*100:.2f}')


Hyperparameter Terbaik: {'fit_prior': True, 'alpha': 0.0001}

Akurasi Validasi Terbaik: 53.12


In [73]:
test_accuracy = best_model_cnb.score(X_test,y_test)
train_accruacy = best_model_cnb.score(X_train,y_train)

y_pred_test = best_model_cnb.predict(X_test)
y_prob_test = best_model_cnb.predict_proba(X_test)

y_pred_train = best_model_cnb.predict(X_train)
y_prob_train = best_model_cnb.predict_proba(X_train)

print(f'\nAkurasi Pada Data Test: {test_accuracy*100:.2f}')
print(f'\nAkurasi Pada Data Train: {train_accruacy*100:.2f}')


Akurasi Pada Data Test: 58.50

Akurasi Pada Data Train: 54.75


In [74]:
def evaluate_model(pred,test,prob,evaluate,model_name='Decision Tree'):
    accuracy = accuracy_score(test,pred)
    precision = precision_score(test, pred, average='macro')
    recall = recall_score(test, pred, average='macro')
    f1 = f1_score(test, pred, average='macro')
    roc_auc = roc_auc_score(test,prob,multi_class='ovr', average='macro')
    logloss = log_loss(test,prob)

    data = {
        'Model': [model_name],
        'Evaluated': [evaluate],
        'Accuracy': [accuracy],
        'Precision': [precision],
        'Recall': [recall],
        'F1-Score': [f1],
        'ROC-AUC': [roc_auc],
        'Log Loss': [logloss]
    }

    df_result = pd.DataFrame(data)
    return df_result

In [75]:
def check_model_fit(df_eval_train,df_eval_test):
    df_combined = pd.concat([df_eval_train,df_eval_test],ignore_index=True)
    accuracy_train = df_eval_train['Accuracy'].values[0]
    accuracy_test = df_eval_test['Accuracy'].values[0]
    gap = accuracy_train - accuracy_test

    if accuracy_train < 0.60 and accuracy_test <0.60:
        status = 'Undeerfitting (Akurasi Rendah)'
    elif gap > 0.05:
        status = f'Overfitting (gap:{gap*100:.2f})'
    elif gap < -0.05:
        status = 'Overfitting / Data Leakage (Test > Train)'
    else:
        status = 'Good Fit'

    df_combined['Status'] = status
    return df_combined

In [76]:
df_eval_train = evaluate_model(y_pred_train,y_train,y_prob_train,evaluate='Training')
df_eval_test = evaluate_model(y_pred_test,y_test,y_prob_test,evaluate='Test')
df_eval = check_model_fit(df_eval_train,df_eval_test)

print('================================= PREDIKSI DENGAN SAMPLE DATASET ===================================')
print(df_eval.to_string())
print("\n" + "="*100 + "\n")

================================= PREDIKSI DENGAN SAMPLE DATASET ===================================
           Model Evaluated  Accuracy  Precision    Recall  F1-Score   ROC-AUC  Log Loss                          Status
0  Decision Tree  Training    0.5475   0.411835  0.310005  0.274786  0.682500  1.020638  Undeerfitting (Akurasi Rendah)
1  Decision Tree      Test    0.5850   0.499244  0.324711  0.300701  0.626068  1.046861  Undeerfitting (Akurasi Rendah)


